<a href="https://colab.research.google.com/github/TCU-DCDA/WRIT20833_2026/blob/main/notebooks/codeAlongs/WRIT20833_Count_Your_Data_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# WRIT 20833 — Count Your Data

**When Coding Meets Culture: Developing Data-Driven Opinions**

Make an editable copy of this notebook by going to **File > Save a copy in Drive**

**New to Colab, or submitting?** See [Working in Colab](https://tcu-dcda.github.io/WRIT20833_2026/guides/working-in-colab.html).

This notebook is for you if you have a **clean CSV of your own data**: the file the YouTube Comment
Collector saved (its text is in `comment_clean`) or the one Clean Your Scraped Data saved (`text_clean`).

On Monday, VADER will score every comment in that file as positive, negative, or neutral. Before a
tool tells you what your comments *feel* like, find out what they *say*. You'll count the words with
the same tools you built in HW2, read the top words where people actually used them, and then write
down what you expect VADER to find. Next week you'll check those predictions against the scores.

**How this notebook works.** The code cells come in three kinds:

- **Edit this one.** Your choices go here. There are three of these cells.
- **Read this one.** It uses things you already know: functions, loops, `Counter`, pandas. Read it
  and make sure you could explain it.
- **Plumbing.** It loads your file or checks for mistakes. Run it; you don't need to read every line.

## Why we're doing this

- **From the "Analysis is recursive" map:** you just finished the Clean loop. This is Count, on your
  own data instead of the course corpus. Each loop starts the same way: predict, run, read closely, revise.
- **Where you'll use it:** HW3 asks where VADER's score disagrees with your reading. You can only
  notice that if you wrote your reading down first. Your predictions also come back in
  "Predictions on the record" (Fri 11/20) and in Reflection 2.
- **The question to keep asking:** what does counting flatten? A word list can't tell you who said
  it, how many times one person said it, or whether they meant it.

# Step 1: Guess Before You Look

Same habit as Term Frequency Part 1: **guess first, then run.** You know your data better than anyone
in the room, because you picked it. Without opening the file, what do you think the most common
meaningful word will be? (Meaningful means not a stopword like "the" or "and.")

In [ ]:
# EDIT THIS ONE: your guesses, before you count anything
MY_GUESS_TOP_WORD = ""        # e.g. "love"
MY_GUESS_WHY = ""             # one short reason, e.g. "the video is a love song"

# Step 2: Upload and Load Your Clean File

Run the first cell below. A **Choose Files** button appears under it: click it and pick your clean
CSV (the one that ends in `.csv`, not the `_info.json` record).

**Colab forgets uploaded files** when your session ends. If you come back later, run the upload
cell again.

**No button?** Some browsers block it. Upload with the **Files** icon (a folder) in the left sidebar
instead, then in the first cell replace everything below the comment with one line naming your file,
like `MY_FILE = "youtube_comments_abc123.csv"`.

In [ ]:
# PLUMBING: click "Choose Files" and pick your clean CSV
from google.colab import files

uploaded = files.upload()          # waits for you to choose a file; returns {file name: contents}
if len(uploaded) == 0:
    raise ValueError("No file was chosen. Run this cell again and pick your clean CSV.")

MY_FILE = list(uploaded.keys())[0]
if len(uploaded) > 1:
    print(f"Warning: You chose {len(uploaded)} files; this notebook uses only the first one.")
print("Uploaded:", MY_FILE)

In [ ]:
# PLUMBING: read your file into a table called my_df and find its text column
import os
import re
from collections import Counter

import pandas as pd

MY_FILE = MY_FILE.strip()
if not os.path.exists(MY_FILE):
    files_here = []
    for name in sorted(os.listdir(".")):
        if name.lower().endswith(".csv"):
            files_here.append(name)
    raise FileNotFoundError(f"Can't find a file named {MY_FILE!r}. CSV files uploaded so far: "
                            f"{files_here or 'none'}. Check the spelling, or upload the file again.")

# the same call HW3's setup cell uses: a comment that says "NA" stays text
my_df = pd.read_csv(MY_FILE, keep_default_na=False)

MY_TEXT_COLUMN = ""   # leave blank to find it automatically
if MY_TEXT_COLUMN == "":
    for candidate in ["comment_clean", "text_clean"]:
        if candidate in my_df.columns:
            MY_TEXT_COLUMN = candidate
            break
if MY_TEXT_COLUMN not in my_df.columns:
    raise ValueError(f"Can't tell which column holds your text. Your columns are {list(my_df.columns)}. "
                     "Type the right one between the quotes in MY_TEXT_COLUMN above and run this cell again.")

my_df[MY_TEXT_COLUMN] = my_df[MY_TEXT_COLUMN].astype(str)
print("DATA SOURCE:", MY_FILE)
print("Text column:", MY_TEXT_COLUMN)
print("Comments:", len(my_df))
print()
print("First one:", my_df[MY_TEXT_COLUMN].iloc[0][:200])

# Step 3: Your Tools from HW2

These are the tools you built in Term Frequency and finished in HW2 Part A: the word splitter, the
stopword list, and `top_meaningful_words`. Same code, nothing new. That's the point: you wrote them
for the course corpus, and now they work on data nobody in the class has seen.

In [ ]:
# READ THIS ONE: the same tools as HW2
def split_into_words(any_chunk_of_text):
    lowercase_text = any_chunk_of_text.lower()
    # split on any run of non-letter/number characters (spaces, punctuation, newlines)
    return re.split(r"\W+", lowercase_text)

stopwords = ["i", "me", "my", "myself", "we", "our", "ours", "ourselves", "you", "your", "yours",
             "yourself", "yourselves", "he", "him", "his", "himself", "she", "her", "hers", "herself",
             "it", "its", "itself", "they", "them", "their", "theirs", "themselves", "what", "which",
             "who", "whom", "this", "that", "these", "those", "am", "is", "are", "was", "were", "be",
             "been", "being", "have", "has", "had", "having", "do", "does", "did", "doing", "a", "an",
             "the", "and", "but", "if", "or", "because", "as", "until", "while", "of", "at", "by",
             "for", "with", "about", "against", "between", "into", "through", "during", "before",
             "after", "above", "below", "to", "from", "up", "down", "in", "out", "on", "off", "over",
             "under", "again", "further", "then", "once", "here", "there", "when", "where", "why",
             "how", "all", "any", "both", "each", "few", "more", "most", "other", "some", "such", "no",
             "nor", "not", "only", "own", "same", "so", "than", "too", "very", "s", "t", "can", "will",
             "just", "don", "should", "now", "ve", "ll", "amp"]

def top_meaningful_words(text, n):
    meaningful_words = []
    for word in split_into_words(text):
        if word != "" and word not in stopwords:
            meaningful_words.append(word)
    return Counter(meaningful_words).most_common(n)

print("Tools ready.")

# Step 4: Count Every Comment at Once

`top_meaningful_words` takes **one** string. Your comments are a whole column of strings. So first,
glue the column into one long text, the same way String Methods glued a list of words back together.

**The structure:**
```python
all_text = " ".join(my_df["column_name"].tolist())
```
- `my_df["column_name"].tolist()` turns the column into a plain Python list of strings (Pandas 02).
- `" ".join(...)` glues that list into one string with a space between comments (String Methods).
- After that, `all_text` is just a long string, so `top_meaningful_words` works on it unchanged.

In [ ]:
# READ THIS ONE: glue the comments together, count, and compare with your guess
all_text = " ".join(my_df[MY_TEXT_COLUMN].tolist())

top_words = top_meaningful_words(all_text, 20)
for word, count in top_words:
    print(f"{count:6}  {word}")

print()
print("Your guess:", MY_GUESS_TOP_WORD or "(none written in Step 1)")
print("Actual top word:", top_words[0][0])

Look down the list before you go on. Which words did you expect? Which surprise you? Are some of them
clutter rather than meaning: the name of the video or channel, "video," "lol," a number, a word
split oddly like "don" from "don't"?

# Step 5: Your Own Stopwords

Same move as HW2 Part C: add a few stopwords of your own and count again. This is a choice, so
make it on purpose. Removing the channel's name is housekeeping; removing "hate" because it's
uncomfortable would change what your data says.

**The structure (from HW2 C1):**
```python
custom_stopwords = stopwords + ["word one", "word two"]
```
- `+` joins two lists into one longer list.
- Then count with the same loop as `top_meaningful_words`, checking `custom_stopwords` instead.

In [ ]:
# EDIT THIS ONE: words to skip for YOUR data (lowercase), then run
MY_EXTRA_STOPWORDS = []       # e.g. ["video", "song", "lol"]

In [ ]:
# READ THIS ONE: count again, skipping your extra words too
custom_stopwords = stopwords + MY_EXTRA_STOPWORDS

meaningful_words = []
for word in split_into_words(all_text):
    if word != "" and word not in custom_stopwords:
        meaningful_words.append(word)

top_words = Counter(meaningful_words).most_common(20)
for word, count in top_words:
    print(f"{count:6}  {word}")
print()
print("Skipped as your own stopwords:", MY_EXTRA_STOPWORDS or "(none)")

# Step 6: Words or People?

A count says how many **times** a word appears, not how many **people** used it. One commenter who
writes "love love LOVE" adds three. So for each top word, check how many comments contain it at all.

This uses the filter from Pandas 02: `my_df[my_df["column_name"].str.lower().str.contains("word", regex=False)]`
keeps only the rows whose text includes that word, and `len(...)` counts those rows.

One honest limit: `.str.contains("love")` also matches "glove" and "lovely." It's a quick check, not
an exact one.

In [ ]:
# READ THIS ONE: for each top word, times used vs. comments that use it
lowercase_text = my_df[MY_TEXT_COLUMN].str.lower()

print(f"{'word':>14}  {'times':>6}  {'comments':>8}")
for word, count in top_words[:10]:
    comments_with_word = my_df[lowercase_text.str.contains(word, regex=False)]
    print(f"{word:>14}  {count:6}  {len(comments_with_word):8}")

print()
print("Total comments:", len(my_df))

Where the two numbers are far apart, a few people are repeating themselves (or one comment is very
long). Where they're close, the word is spread across many voices. Which kind are your top words?

# Step 7: Read Closely

Counting tells you *what* shows up, not *how* it was meant. Pick one of your top words and read the
comments that use it. Look for the things a word count can't see: sarcasm, quoting the other side,
a joke, a question, a word used two ways.

In [ ]:
# EDIT THIS ONE: one word from your top 20 to read in context
WORD_TO_READ = ""             # e.g. "love"

In [ ]:
# READ THIS ONE: up to eight comments that use your word, printed in full
WORD_TO_READ = WORD_TO_READ.strip().lower()
if WORD_TO_READ == "":
    WORD_TO_READ = top_words[0][0]
    print(f"(No word chosen, so using your top word, {WORD_TO_READ!r}.)")
    print()

matches = my_df[lowercase_text.str.contains(WORD_TO_READ, regex=False)]
print(f"{len(matches)} comments use {WORD_TO_READ!r}. Showing up to eight at random:")
print()
for comment in matches[MY_TEXT_COLUMN].sample(n=min(8, len(matches)), random_state=1):
    print("-", comment)
    print()

Change `random_state=1` to another number and run the cell again to see a different eight. Try a
second word too.

# Step 8: Predictions on the Record

## What VADER is (the short version)

On Monday you'll use **VADER**, a **sentiment analysis** tool. Its job is to read a piece of text
and say how positive or negative it sounds. Researchers built it for short social media posts like
yours. Here's all you need to know for now:

- **It starts from a word list.** People rated thousands of words for feeling, from very negative to
  very positive. "Love" and "great" score positive, "hate" and "awful" negative, and most words,
  like "school" or "law," score zero.
- **It adds a few rules.** ALL CAPS and "!!!" make a feeling stronger. "Not" flips it ("not good"
  reads as negative). "But" gives more weight to what comes after it.
- **Emoji are hit or miss.** VADER swaps each emoji for its name and scores those words. So 😂
  ("face with tears of joy") counts as positive and 😢 ("crying face") as negative, but 😡
  ("pouting face") and 👍 ("thumbs up") score zero, and 🔥 ("fire") counts as negative.
- **It gives each comment one score,** from -1 (most negative) to +1 (most positive), and labels the
  comment **positive**, **negative**, or **neutral**.
- **It reads words, not meaning.** It doesn't know who is speaking, what the video was about, or
  whether someone is joking.

## Your predictions

Now write down what you expect VADER to find on Monday. You don't need to know more about how it
works than what's above. This is *your* reading, on the record before any score exists. Wrong
predictions are useful. They show you where a tool and a careful reader part ways.

Double-click each cell below and write under the line. Your answers save with your Drive copy.

**Prediction 1: the words** *(double-click to edit; write your answer below the line)*

Look at your top 20 words. Which ones do you think a sentiment tool will count as **positive**? Which as **negative**? Which will it treat as neutral even though, in your comments, they carry feeling?

---

**Prediction 2: the overall lean** *(double-click to edit; write your answer below the line)*

Will your comments come out mostly positive, mostly negative, or split? Give a rough guess for the share that will score positive, and say what in Steps 4–7 makes you think so.

---

**Prediction 3: a comment VADER will get wrong** *(double-click to edit; write your answer below the line)*

Copy one comment from Step 7 (or anywhere in your data) that you think a sentiment tool will misread. What will it miss: sarcasm, a quote of the other side, a joke, an emoji, slang, something else?

---

**Prediction 4: what counting missed** *(double-click to edit; write your answer below the line)*

Name one thing you noticed while reading in Step 7 that the word counts in Steps 4–6 couldn't show you.

---

# Where This Goes Next

**Monday (Day 13):** the VADER code-along shows how a sentiment score is made. Then HW3 opens. In its
setup cell, upload the same CSV and fill in the two "Choose your data" lines:

```python
MY_CSV = "your_file.csv"
MY_TEXT_COLUMN = "comment_clean"      # or "text_clean" if you used Clean Your Scraped Data
```

**Keep this notebook.** When HW3 gives you VADER's scores, come back to Step 8 and check each
prediction. You'll bring what you find to "Predictions on the record" on Friday 11/20, and it's
good evidence for Reflection 2, due the same day.

# Playground

Some things to try:
- **Go further down.** Change `20` to `50` in Step 5. What shows up in the long tail that the top
  20 hid?
- **Look for an absence.** Pick a word you expected to see and didn't. Use Step 7's cell to check
  whether anyone used it at all.
- **Compare with the course corpus.** In HW2, the Ten Commandments comments' top meaningful words
  were "commandments," "religion," "god," "children," and "schools." How different is the
  vocabulary of your conversation?

In [ ]:
# use this space to experiment!